## Example 1: Basic Geographically Weighted Regression setup

In [ ]:
import mgwr
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

# Example 1: Basic Geographically Weighted Regression setup
print("mgwr module imported successfully")

try:
    print("Example 1 - Basic Geographically Weighted Regression setup")
    
    # Generate synthetic spatial data for regression analysis
    np.random.seed(42)
    n_points = 100
    
    # Create spatial coordinates (simulating geographic locations)
    coords = np.random.uniform(0, 10, size=(n_points, 2))
    
    # Generate spatially varying relationships
    # Coefficient varies across space based on location
    x1 = np.random.normal(0, 1, n_points)
    x2 = np.random.normal(0, 1, n_points)
    
    # Create spatially varying coefficients
    beta0 = 2 + 0.3 * coords[:, 0]  # Intercept varies with x-coordinate
    beta1 = 1 + 0.2 * coords[:, 1]  # Slope varies with y-coordinate
    beta2 = 0.5 - 0.1 * coords[:, 0] * coords[:, 1]  # Interaction effect
    
    # Generate dependent variable with spatial variation
    # mgwr requires the dependent variable as a COLUMN vector (n, 1). Passing the 1-D (n,) array
    # gets as far as the parallel bandwidth search and then dies inside a joblib worker with
    # `IndexError: invalid index to scalar variable` — a message that names neither y nor its shape.
    y = (beta0 + beta1 * x1 + beta2 * x2 + np.random.normal(0, 0.5, n_points)).reshape(-1, 1)
    
    # Prepare data matrices
    X = np.column_stack([x1, x2])  # Independent variables
    
    print(f"Generated spatial dataset:")
    print(f"  Number of observations: {n_points}")
    print(f"  Coordinate bounds: X[{coords[:, 0].min():.2f}, {coords[:, 0].max():.2f}], Y[{coords[:, 1].min():.2f}, {coords[:, 1].max():.2f}]")
    print(f"  Independent variables: {X.shape[1]}")
    print(f"  Y variable range: [{y.min():.2f}, {y.max():.2f}]")
    
    # Bandwidth selection for GWR
    print(f"\nBandwidth Selection:")
    bw_selector = Sel_BW(coords, y, X)
    
    # Find optimal bandwidth using AICc criterion
    optimal_bw = bw_selector.search(criterion='AICc')
    print(f"  Optimal bandwidth: {optimal_bw:.3f}")
    print(f"  Bandwidth type: {'Fixed distance' if bw_selector.fixed else 'Adaptive (k-nearest neighbors)'}")
    # Sel_BW is adaptive by default, so the bandwidth is a neighbour count between k and n.
    assert not bw_selector.fixed and X.shape[1] + 1 <= optimal_bw <= n_points, optimal_bw
    
    # Fit basic GWR model
    gwr_model = GWR(coords, y, X, bw=optimal_bw)
    gwr_results = gwr_model.fit()
    
    print(f"\nGWR Model Results:")
    # The parameter count (intercept included) is `k`.
    print(f"  Number of parameters: {gwr_results.k}")
    print(f"  Effective degrees of freedom: {gwr_results.tr_S:.2f}")
    # The results object exposes `sigma2`, the residual variance.
    print(f"  Sigma (residual standard error): {gwr_results.sigma2 ** 0.5:.3f}")
    print(f"  AICc: {gwr_results.aicc:.3f}")
    print(f"  R-squared: {gwr_results.localR2.mean():.3f} (mean local)")
    
    # Analyze coefficient variation
    print(f"\nCoefficient Spatial Variation:")
    param_names = ['Intercept', 'X1', 'X2']
    for i, name in enumerate(param_names):
        coef_values = gwr_results.params[:, i]
        print(f"  {name}: mean={coef_values.mean():.3f}, std={coef_values.std():.3f}, range=[{coef_values.min():.3f}, {coef_values.max():.3f}]")
    assert gwr_results.k == 3 and gwr_results.params.shape == (n_points, 3), gwr_results.params.shape
    assert X.shape[1] < gwr_results.tr_S < n_points and np.isfinite(gwr_results.aicc), (gwr_results.tr_S, gwr_results.aicc)
    assert ((gwr_results.localR2 > 0) & (gwr_results.localR2 <= 1)).all()
    
    print("mgwr example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mgwr example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Cross-validation and model diagnostics for GWR

In [ ]:
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

# Example 2: Cross-validation and model diagnostics for GWR
print("mgwr cross-validation and model diagnostics")

try:
    print("Example 2 - Cross-validation and model diagnostics for GWR")
    
    # Generate synthetic spatial data with known relationships
    np.random.seed(456)
    n_points = 80
    
    # Create spatial coordinates in a rectangular grid
    x_coords = np.repeat(np.linspace(0, 8, 10), 8)
    y_coords = np.tile(np.linspace(0, 8, 8), 10)
    coords = np.column_stack([x_coords, y_coords])
    
    # Generate independent variables with spatial correlation
    pop_density = 100 + 50 * np.random.normal(0, 1, n_points) + 10 * coords[:, 0]
    employment_rate = 0.7 + 0.1 * np.random.normal(0, 1, n_points) - 0.02 * coords[:, 1]
    house_age = 20 + 30 * np.random.exponential(1, n_points)
    
    # Create spatially varying coefficients
    beta_pop = 0.5 + 0.1 * np.sin(coords[:, 0] * np.pi / 4)  # Population density coefficient
    beta_emp = 200 + 100 * np.cos(coords[:, 1] * np.pi / 4)  # Employment rate coefficient
    beta_age = -2 - 1 * (coords[:, 0] * coords[:, 1]) / 16   # House age coefficient
    
    # Generate dependent variable (housing prices)
    # mgwr requires y as a column vector (n, 1); a 1-D y fails inside the parallel bandwidth search.
    y = (50 + beta_pop * pop_density/100 + beta_emp * employment_rate + 
         beta_age * house_age/10 + np.random.normal(0, 10, n_points)).reshape(-1, 1)
    
    X = np.column_stack([pop_density/100, employment_rate, house_age/10])
    
    print(f"Cross-validation dataset:") 
    print(f"  Observations: {n_points}")
    print(f"  Variables: Population density, Employment rate, House age")
    print(f"  Y range (housing prices): ${y.min():.0f} to ${y.max():.0f}K")
    
    # Bandwidth selection with cross-validation
    print(f"\nBandwidth Selection with Cross-Validation:")
    bw_selector = Sel_BW(coords, y, X)
    
    # Test different bandwidth criteria
    criteria = ['AICc', 'BIC', 'CV']
    bandwidths = {}
    
    for criterion in criteria:
        try:
            bw = bw_selector.search(criterion=criterion)
            bandwidths[criterion] = bw
            print(f"  {criterion} optimal bandwidth: {bw:.3f}")
        except Exception as criterion_error:
            print(f"  {criterion}: calculation failed — {type(criterion_error).__name__}: {criterion_error}")
            raise
    
    assert set(bandwidths) == set(criteria), bandwidths
    assert all(X.shape[1] + 1 <= bw <= n_points for bw in bandwidths.values()), bandwidths

    # Use AICc bandwidth for main analysis
    optimal_bw = bandwidths['AICc']
    
    # Fit GWR model
    gwr_model = GWR(coords, y, X, bw=optimal_bw)
    gwr_results = gwr_model.fit()
    
    print(f"\nModel Diagnostics:")
    print(f"  Bandwidth used: {optimal_bw:.3f}")
    print(f"  Effective degrees of freedom: {gwr_results.tr_S:.2f}")
    # `resid_ss` is the residual sum of squares (`RSS` is the per-location weighted one);
    # `sigma2` is the residual variance.
    print(f"  Residual sum of squares: {gwr_results.resid_ss:.2f}")
    print(f"  Sigma estimate: {gwr_results.sigma2 ** 0.5:.3f}")
    print(f"  Log-likelihood: {gwr_results.llf:.2f}")
    
    # Model fit statistics
    print(f"\nModel Fit Statistics:")
    print(f"  AICc: {gwr_results.aicc:.2f}")
    print(f"  BIC: {gwr_results.bic:.2f}") 
    print(f"  Mean local R-squared: {gwr_results.localR2.mean():.3f}")
    print(f"  R-squared range: [{gwr_results.localR2.min():.3f}, {gwr_results.localR2.max():.3f}]")
    assert gwr_results.params.shape == (n_points, 4), gwr_results.params.shape
    assert X.shape[1] < gwr_results.tr_S < n_points, gwr_results.tr_S
    assert all(np.isfinite(v) for v in (gwr_results.aicc, gwr_results.bic, gwr_results.llf))
    assert ((gwr_results.localR2 > 0) & (gwr_results.localR2 <= 1)).all()
    
    # Residual analysis
    residuals = gwr_results.resid_response
    standardized_residuals = residuals / (gwr_results.sigma2 ** 0.5)
    
    print(f"\nResidual Analysis:")
    print(f"  Mean residual: {residuals.mean():.3f} (should be ≈ 0)")
    print(f"  Residual std: {residuals.std():.3f}")
    print(f"  Standardized residuals outside [-2, 2]: {np.sum(np.abs(standardized_residuals) > 2)}")
    print(f"  Moran's I test for residual autocorrelation available via pysal")
    
    # Parameter stability across space
    print(f"\nParameter Spatial Stability:")
    param_names = ['Intercept', 'Pop_Density', 'Employment', 'House_Age']
    for i, name in enumerate(param_names):
        params = gwr_results.params[:, i]
        stability = params.std() / abs(params.mean()) if abs(params.mean()) > 1e-6 else float('inf')
        stability_level = "High" if stability > 0.5 else ("Medium" if stability > 0.2 else "Low")
        print(f"  {name}: CV = {stability:.3f} ({stability_level} spatial variation)")
    
    print("mgwr example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mgwr example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Real-world application with spatial data analysis

In [ ]:
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

# Example 3: Real-world application with spatial data analysis
print("mgwr real-world spatial data analysis")

try:
    print("Example 3 - Real-world application: Urban housing price analysis")
    
    # Simulate real-world urban housing data
    np.random.seed(789)
    n_locations = 150
    
    # Create urban-like spatial structure (city center + suburbs)
    # City center coordinates
    center_x, center_y = 5, 5
    
    # Generate locations with urban clustering
    angles = np.random.uniform(0, 2*np.pi, n_locations)
    distances = np.random.exponential(2, n_locations)  # Exponential decay from center
    
    x_coords = center_x + distances * np.cos(angles) + np.random.normal(0, 0.5, n_locations)
    y_coords = center_y + distances * np.sin(angles) + np.random.normal(0, 0.5, n_locations)
    coords = np.column_stack([x_coords, y_coords])
    
    # Distance from city center
    dist_center = np.sqrt((x_coords - center_x)**2 + (y_coords - center_y)**2)
    
    # Generate realistic urban variables
    # School quality (higher in suburbs, varies by district)
    school_quality = 7 + 2 * np.sin(x_coords * np.pi / 3) * np.cos(y_coords * np.pi / 4) + np.random.normal(0, 0.5, n_locations)
    school_quality = np.clip(school_quality, 1, 10)  # Rating 1-10
    
    # Crime rate (higher in city center, varies by neighborhood)
    crime_rate = 5 + 3 * np.exp(-dist_center/2) + 2 * np.random.exponential(0.5, n_locations)
    crime_rate = np.clip(crime_rate, 0.1, 20)  # Crimes per 1000 residents
    
    # Public transit access (better near center)
    transit_access = 8 * np.exp(-dist_center/3) + np.random.gamma(2, 1, n_locations)
    transit_access = np.clip(transit_access, 0, 10)  # Access score 0-10
    
    # Create spatially varying housing price relationships
    # Base price varies by location (proximity premium)
    base_price = 300 + 50 * (10 - dist_center) * np.exp(-dist_center/4)
    
    # School quality premium (varies across city - some areas value it more)
    school_premium = (20 + 10 * np.sin(x_coords * np.pi / 5)) * (school_quality - 5)
    
    # Crime discount (varies by neighborhood tolerance)
    crime_discount = -(10 + 5 * np.cos(y_coords * np.pi / 6)) * np.log(crime_rate)
    
    # Transit premium (varies by car ownership patterns)
    transit_premium = (5 + 3 * np.cos((x_coords + y_coords) * np.pi / 8)) * np.sqrt(transit_access)
    
    # Final house prices (in thousands)
    house_prices = base_price + school_premium + crime_discount + transit_premium + np.random.normal(0, 25, n_locations)
    house_prices = np.maximum(house_prices, 50)  # Minimum $50k
    
    X = np.column_stack([school_quality, np.log(crime_rate)])
    # mgwr requires the dependent variable as a COLUMN vector (n, 1) — a 1-D array dies inside the
    # parallel bandwidth search with `IndexError: invalid index to scalar variable`.
    y = np.asarray(house_prices).reshape(-1, 1)
    
    print(f"Urban Housing Analysis Dataset:")
    print(f"  Locations analyzed: {n_locations}")
    print(f"  Spatial extent: {coords[:, 0].min():.1f} to {coords[:, 0].max():.1f} km")
    print(f"  Variables: School quality, Crime rate (log)")
    print(f"  House prices: ${y.min():.0f}K to ${y.max():.0f}K (mean: ${y.mean():.0f}K)")
    
    # Compare global vs local models
    print(f"\nModel Comparison:")
    
    # Global OLS model (for comparison), fitted with numpy least squares
    A = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    global_predictions = A @ beta
    global_r2 = 1 - ((y - global_predictions)**2).sum() / ((y - y.mean())**2).sum()
    global_rmse = np.sqrt(np.mean((y - global_predictions)**2))
    
    print(f"  Global OLS Model:")
    print(f"    R-squared: {global_r2:.3f}")
    print(f"    RMSE: ${global_rmse:.1f}K")
    print(f"    Coefficients: {beta[1:].ravel()}")
    assert 0 < global_r2 < 1, global_r2
    
    # Geographically Weighted Regression
    bw_selector = Sel_BW(coords, y, X)
    optimal_bw = bw_selector.search(criterion='AICc')
    
    gwr_model = GWR(coords, y, X, bw=optimal_bw)
    gwr_results = gwr_model.fit()
    
    print(f"  Geographically Weighted Regression:")
    print(f"    Bandwidth: {optimal_bw:.0f} nearest neighbours")
    print(f"    Mean local R-squared: {gwr_results.localR2.mean():.3f}")
    print(f"    RMSE: ${np.sqrt(gwr_results.resid_ss / n_locations):.1f}K")
    print(f"    AICc: {gwr_results.aicc:.1f}")
    assert gwr_results.params.shape == (n_locations, 3) and np.isfinite(gwr_results.aicc)
    assert X.shape[1] + 1 <= optimal_bw <= n_locations, optimal_bw
    
    # Spatial analysis of coefficients
    print(f"\nSpatial Variation in Relationships:")
    coef_names = ['Intercept', 'School_Quality', 'Crime_Rate(log)']
    
    for i, name in enumerate(coef_names):
        coef_vals = gwr_results.params[:, i]
        coef_range = coef_vals.max() - coef_vals.min()
        coef_std = coef_vals.std()
        
        print(f"  {name}:")
        print(f"    Mean coefficient: {coef_vals.mean():.2f}")
        print(f"    Spatial range: {coef_range:.2f}")
        print(f"    Spatial std: {coef_std:.2f}")
        
        # Identify locations with extreme coefficients
        high_coef = coef_vals > (coef_vals.mean() + coef_std)
        low_coef = coef_vals < (coef_vals.mean() - coef_std)
        print(f"    High coefficient areas: {np.sum(high_coef)} locations")
        print(f"    Low coefficient areas: {np.sum(low_coef)} locations")
    
    # Model validation metrics
    predictions = gwr_results.predy.flatten()
    residuals = y.ravel() - predictions
    assert predictions.shape == (n_locations,)
    # GWR fits each location locally, so its in-sample error is below the global OLS error here.
    assert np.sqrt(np.mean(residuals**2)) < global_rmse, (np.sqrt(np.mean(residuals**2)), global_rmse)
    
    print(f"\nModel Performance:")
    print(f"  Mean Absolute Error: ${np.mean(np.abs(residuals)):.1f}K")
    print(f"  Median Absolute Error: ${np.median(np.abs(residuals)):.1f}K") 
    print(f"  90th percentile error: ${np.percentile(np.abs(residuals), 90):.1f}K")
    
    # Local model performance varies across space
    local_performance = gwr_results.localR2
    print(f"  Local R-squared distribution:")
    print(f"    25th percentile: {np.percentile(local_performance, 25):.3f}")
    print(f"    50th percentile: {np.percentile(local_performance, 50):.3f}")
    print(f"    75th percentile: {np.percentile(local_performance, 75):.3f}")
    
    print(f"\nPractical Insights:")
    print(f"  - GWR reveals spatial heterogeneity in housing markets")
    print(f"  - School quality premium varies across neighborhoods")
    print(f"  - Crime impact differs by local market conditions")
    print(f"  - Model useful for location-specific price predictions")
    print(f"  - Local R² > 0.6 in {np.sum(local_performance > 0.6)} of {n_locations} areas")
    
    print("mgwr example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mgwr example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: mgwr advanced capabilities and production workflow

In [ ]:
import importlib
import mgwr
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW
import numpy as np

# Example 4: mgwr advanced capabilities and production workflow
print("mgwr advanced capabilities and production workflow")

try:
    print("Example 4 - Advanced mgwr capabilities and production workflow")
    
    # Demonstrate package structure and capabilities
    print(f"mgwr Package Overview:")
    print(f"  Module type: {type(mgwr).__name__}")
    print(f"  Package location: {mgwr.__file__ if hasattr(mgwr, '__file__') else 'Built-in'}")
    
    # Key classes and functions, each from the module that defines it (Sel_BW lives in mgwr.sel_bw, not mgwr.gwr)
    key_components = {
        'GWR': ('mgwr.gwr', 'Basic Geographically Weighted Regression'),
        'MGWR': ('mgwr.gwr', 'Multiscale Geographically Weighted Regression'), 
        'Sel_BW': ('mgwr.sel_bw', 'Bandwidth selection algorithms')
    }
    
    print(f"  Key Components:")
    for component, (module_name, description) in key_components.items():
        component_class = getattr(importlib.import_module(module_name), component)
        assert isinstance(component_class, type), f"{module_name}.{component} is not a class"
        print(f"    ✓ {component} ({module_name}): {description}")
    
    # Production workflow demonstration
    print(f"\nProduction Workflow Pattern:")
    
    # Step 1: Data preparation and validation
    np.random.seed(1000)
    n_obs = 100
    coords = np.random.uniform(0, 10, size=(n_obs, 2))
    X = np.random.normal(0, 1, size=(n_obs, 3))
    # Column vector (n, 1), as mgwr requires everywhere.
    y = (5 + X @ [1.5, -0.8, 0.3] + np.random.normal(0, 0.5, n_obs)).reshape(-1, 1)
    
    print(f"  1. Data Validation:")
    print(f"     ✓ Coordinates shape: {coords.shape}")
    print(f"     ✓ Features shape: {X.shape}")  
    print(f"     ✓ Target shape: {y.shape}")
    print(f"     ✓ No missing values: {not (np.isnan(X).any() or np.isnan(y).any())}")
    
    # Step 2: Bandwidth selection with multiple criteria
    print(f"  2. Bandwidth Selection:")
    bw_selector = Sel_BW(coords, y, X)
    
    # Test multiple bandwidth selection methods: every criterion must return a bandwidth
    bw_methods = {'AICc': None, 'BIC': None, 'CV': None}
    for method in bw_methods:
        bw = bw_selector.search(criterion=method)
        bw_methods[method] = bw
        print(f"     ✓ {method} bandwidth: {bw:.3f}")
    assert all(X.shape[1] + 1 <= bw <= n_obs for bw in bw_methods.values()), bw_methods

    optimal_bw = bw_methods['AICc']

    # Step 3: Model fitting - the fit this example exists for; a failure here fails the cell
    print(f"  3. Model Fitting:")
    gwr_model = GWR(coords, y, X, bw=optimal_bw)
    gwr_results = gwr_model.fit()
    print(f"     ✓ GWR model fitted successfully")
    print(f"     ✓ Effective DoF: {gwr_results.tr_S:.2f}")
    print(f"     ✓ AICc: {gwr_results.aicc:.2f}")
    assert X.shape[1] < gwr_results.tr_S < n_obs and np.isfinite(gwr_results.aicc), (gwr_results.tr_S, gwr_results.aicc)

    # Step 4: Model diagnostics and validation
    if gwr_results:
        print(f"  4. Model Diagnostics:")
        
        # Check model assumptions
        residuals = gwr_results.resid_response
        studentized_residuals = residuals / (gwr_results.sigma2 ** 0.5)
        
        diagnostics = {
            'Mean residual near zero': abs(residuals.mean()) < 0.1,
            'No extreme outliers': np.sum(np.abs(studentized_residuals) > 3) < n_obs * 0.05,
            'Reasonable local R²': gwr_results.localR2.mean() > 0.1,
            'Model converged': gwr_results.aicc < float('inf')
        }
        
        for check, passed in diagnostics.items():
            status = "✓" if passed else "⚠"
            print(f"     {status} {check}")
        assert all(diagnostics.values()), diagnostics  # all four hold on this seeded data
        
        # Performance metrics; `resid_ss` is the residual sum of squares
        rmse = np.sqrt(gwr_results.resid_ss / n_obs)
        mae = np.mean(np.abs(residuals))
        
        print(f"     Performance: RMSE={rmse:.3f}, MAE={mae:.3f}")
    
    # Step 5: Advanced features
    print(f"  5. Advanced Capabilities:")
    
    # Multiscale GWR: the backfitting search picks one bandwidth per column (intercept included)
    mgwr_selector = Sel_BW(coords, y, X, multi=True)
    mgwr_bws = mgwr_selector.search()
    print(f"     ✓ MGWR bandwidths (intercept, x1, x2, x3): {mgwr_bws}")
    assert len(mgwr_bws) == X.shape[1] + 1 and all(0 < bw <= n_obs for bw in mgwr_bws), mgwr_bws
    mgwr_results = MGWR(coords, y, X, mgwr_selector).fit()
    print(f"     ✓ MGWR fitted: AICc={mgwr_results.aicc:.2f}")
    assert mgwr_results.params.shape == (n_obs, X.shape[1] + 1) and np.isfinite(mgwr_results.aicc)
    
    # Additional capabilities
    capabilities = [
        "Fixed and adaptive bandwidth kernels",
        "Gaussian and bi-square kernel functions", 
        "Cross-validation for bandwidth selection",
        "Model comparison via information criteria",
        "Local parameter estimation and inference",
        "Residual analysis and diagnostics"
    ]
    
    print(f"     Additional Features:")
    for capability in capabilities:
        print(f"       • {capability}")
    
    # Step 6: Best practices
    print(f"  6. Production Best Practices:")
    best_practices = [
        "Validate spatial data quality and coordinate systems",
        "Test multiple bandwidth selection criteria", 
        "Check model assumptions and residual patterns",
        "Compare with global regression models",
        "Consider computational complexity for large datasets",
        "Document bandwidth choices and model diagnostics",
        "Cross-validate model performance spatially",
        "Monitor for spatial autocorrelation in residuals"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"     {i}. {practice}")
    
    # Package ecosystem integration
    print(f"\nEcosystem Integration:")
    integrations = [
        "PySAL: Spatial analysis library ecosystem",
        "GeoPandas: Geospatial data manipulation", 
        "Scikit-learn: Machine learning model comparison",
        "Matplotlib: Spatial visualization of results",
        "Shapely: Geometric operations on spatial data"
    ]
    
    for integration in integrations:
        print(f"  • {integration}")
    
    print(f"\nTypical Use Cases:")
    use_cases = [
        "Real estate price modeling with spatial variation",
        "Environmental factor analysis across regions",
        "Economic development impact assessment", 
        "Urban planning and zoning analysis",
        "Public health spatial epidemiology",
        "Transportation accessibility modeling"
    ]
    
    for use_case in use_cases:
        print(f"  • {use_case}")
    
    print("mgwr example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mgwr example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")